# QSVT クイックスタート

このノートブックでは、小さな周期シフト LCU の block encoding に QSVT を適用します。
また、qkernel を先に serialize し、受信側で deserialize してから block encoding を
bind する流れも確認します。

In [1]:
import qamomile.circuit as qmc
from qamomile.circuit.serialization import deserialize, serialize
from qamomile.linalg import PeriodicShiftLCU
from qamomile.qiskit import QiskitTranspiler

## 1. QSVT を使う qkernel

block encoding をグローバル変数として埋め込まず、`block_encoding` 引数として
宣言するのが重要です。これにより、tracing 時点では具体的な encoding を決めずに
qkernel を serialize できます。

`phases` は実行時パラメータとして残します。一方、位相の個数は回路構造を決めるため、
`phase_count` を transpile 時に bind します。

In [2]:
@qmc.qkernel
def qsvt_sample(
    block_encoding: qmc.LCUBlockEncoding,
    phases: qmc.Vector[qmc.Float],
    phase_count: qmc.UInt,
) -> qmc.Vector[qmc.Bit]:
    """Apply QSVT and measure its signal register."""
    signal = qmc.qubit_array(block_encoding.num_signal_qubits, "signal")
    system = qmc.qubit_array(block_encoding.num_system_qubits, "system")
    signal, _ = qmc.qsvt(
        signal,
        system,
        phases,
        block_encoding,
        phase_count=phase_count,
    )
    return qmc.measure(signal)

## 2. tracing 結果を送受信する

ここでは同じプロセス内で往復させていますが、`payload` を別プロセスや別マシンへ
送っても手順は同じです。この時点では LCU の係数も QSVT の位相も未確定です。

In [3]:
payload = serialize(qsvt_sample)
received_kernel = deserialize(payload)

print(f"serialized size: {len(payload)} bytes")

serialized size: 35892 bytes


## 3. 受信側で周期シフト LCU を作る

1 qubit の周期境界上で、$T|x\rangle=|x+1\pmod 2\rangle$ とします。
今回の線形演算子は

$$
A = I + 0.5T
$$

です。係数の絶対値和は $\alpha=1.5$ なので、block encoding の左上ブロックは
$A/\alpha$ を表します。QSVT が変換する特異値も、この正規化された演算子のものです。

In [4]:
periodic_lcu = PeriodicShiftLCU.from_coefficients(
    {0: 1.0, 1: 0.5},
    register_sizes=(1,),
)
block_encoding = qmc.periodic_shift_lcu_block_encoding(periodic_lcu)

print("normalization:", block_encoding.normalization)
print("signal qubits:", block_encoding.num_signal_qubits)
print("system qubits:", block_encoding.num_system_qubits)

normalization: 1.5
signal qubits: 1
system qubits: 1


## 4. block encoding を bind して Qiskit へ transpile する

`block_encoding` と `phase_count` は compile-time の `bindings` に入れます。
`phases` は `parameters` に指定するため、transpile 後も実行時パラメータとして残ります。

In [5]:
phase_count = 3
transpiler = QiskitTranspiler()
executable = transpiler.transpile(
    received_kernel,
    bindings={
        "block_encoding": block_encoding,
        "phase_count": phase_count,
    },
    parameters=["phases"],
)

## 5. 同じ回路を異なる位相で実行する

位相ベクトルの長さは、transpile 時に指定した `phase_count` と一致させます。
下の二つの実行では回路を再度 transpile せず、`phases` だけを差し替えています。

In [6]:
executor = transpiler.executor()
shots = 1024

phase_sets = {
    "zero phases": [0.0, 0.0, 0.0],
    "nonzero phases": [0.15, -0.55, 0.9],
}

for label, runtime_phases in phase_sets.items():
    result = executable.sample(
        executor,
        bindings={"phases": runtime_phases},
        shots=shots,
    ).result()
    print(f"{label}: {result.results}")

zero phases: [((0,), 1024)]
nonzero phases: [((1,), 52), ((0,), 972)]


`result.results` は `(signal の測定値, 出現回数)` の列です。signal register がすべて
0 の結果は、block encoding の射影部分に残ったショットに対応します。

QSVT の位相列は、近似したい多項式に応じて別途合成します。`qmc.qsvt` が受け取るのは
$R(\phi)=\exp(i\phi(2\Pi-I))$ という projector-rotation 規約の位相です。
pyqsp の `Wx` 規約など、別の規約で生成した位相は事前に変換してください。